# Fresh small MRI baseline
GPU recommended. Attach saved notebook 02 output only. A compact CNN starts from scratch, averages slice predictions, and uses masked report-label BCE. No weight downloads. This is a pipeline smoke baseline; validation measures agreement with heuristic labels, not a competition score.

In [ ]:
from pathlib import Path
import json, time, hashlib
import numpy as np
import pandas as pd
ROOTS = ['/kaggle/input/datasets/aymenslimani', '/kaggle/input']
def locate(name):
    hits = sorted(set(p.resolve() for root in ROOTS if Path(root).exists() for p in Path(root).rglob(name)))
    if not hits: raise FileNotFoundError(f'Attach the saved previous notebook output containing {name}')
    if len(hits)>1 and len({hashlib.sha256(p.read_bytes()).hexdigest() for p in hits})>1:
        raise ValueError(f'Conflicting inputs; select one attachment: {hits}')
    return hits[0]
TARGETS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import roc_auc_score
torch.manual_seed(42); np.random.seed(42)
INDEX=locate('fresh_cache_index.csv'); CACHE=INDEX.parent
meta=json.loads((CACHE/'fresh_cache.json').read_text()); FOLD=meta['fold']
df=pd.read_csv(CACHE/'fresh_cached_manifest.csv',dtype={'StudyInstanceUID':str,'group':str})
EPOCHS=3; BATCH_SIZE=8; NUM_WORKERS=0; LR=1e-3
OUT=Path('/kaggle/working/rsna_fresh_baseline'); OUT.mkdir(exist_ok=True)
yc=['y_'+str(i) for i in range(12)]; wc=['w_'+str(i) for i in range(12)]
tr=df[df.fold!=FOLD].reset_index(drop=True); va=df[df.fold==FOLD].reset_index(drop=True)
if tr.empty or va.empty: raise ValueError('Cache must contain train and validation studies')
assert not set(tr.group)&set(va.group)
class MRI(Dataset):
    def __init__(self,frame): self.frame=frame
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]; a=np.load(CACHE/(r.StudyInstanceUID+'.npz'))['images']
        ids=np.linspace(0,len(a)-1,meta['n_slices']).round().astype(int)
        return torch.tensor(a[ids,None]/255.,dtype=torch.float32),torch.tensor(r[yc].to_numpy(dtype=float),dtype=torch.float32),torch.tensor(r[wc].to_numpy(dtype=float),dtype=torch.float32)
train_loader=DataLoader(MRI(tr),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS)
val_loader=DataLoader(MRI(va),batch_size=BATCH_SIZE,num_workers=NUM_WORKERS)
class Baseline(nn.Module):
    def __init__(self):
        super().__init__(); layers=[]; channels=1
        for width in [16,32,64]:
            layers += [nn.Conv2d(channels,width,3,stride=2,padding=1),nn.ReLU(),nn.Conv2d(width,width,3,padding=1),nn.ReLU()]; channels=width
        self.encoder=nn.Sequential(*layers,nn.AdaptiveAvgPool2d(1)); self.head=nn.Linear(64,12)
    def forward(self,x):
        b,s,c,h,w=x.shape
        return self.head(self.encoder(x.reshape(b*s,c,h,w)).flatten(1).reshape(b,s,-1).mean(1))
device='cuda' if torch.cuda.is_available() else 'cpu'; model=Baseline().to(device); opt=torch.optim.AdamW(model.parameters(),lr=LR)
active=torch.tensor(tr[wc].sum().values>0,device=device)
if not active.any(): raise ValueError('No supervised training labels; improve report rules')
print('Device',device,'train',len(tr),'validation',len(va),'active targets',int(active.sum()))
def loss_fn(logits,y,w):
    w=w*active
    return (nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')*w).sum(),w.sum()
history=[]; best=float('inf'); start=time.time()
for epoch in range(EPOCHS):
    model.train(); ts=tw=0.
    for x,y,w in train_loader:
        x,y,w=x.to(device),y.to(device),w.to(device); opt.zero_grad(); num,den=loss_fn(model(x),y,w)
        if den==0: continue
        (num/den).backward(); opt.step(); ts+=num.item(); tw+=den.item()
    model.eval(); vs=vw=0.; predictions=[]
    with torch.no_grad():
        for x,y,w in val_loader:
            logits=model(x.to(device)); num,den=loss_fn(logits,y.to(device),w.to(device)); vs+=num.item(); vw+=den.item(); predictions.append(logits.sigmoid().cpu().numpy())
    if vw==0: raise ValueError('No validation supervision for trained targets')
    value=vs/vw; history.append({'epoch':epoch+1,'train_bce':ts/max(tw,1),'val_bce':value,'seconds':time.time()-start})
    print(history[-1])
    if value<best:
        best=value; torch.save({'state_dict':model.state_dict(),'cache_metadata':meta,'targets':TARGETS,'epoch':epoch+1,'val_bce':best},OUT/'best.pt')
        pred=np.concatenate(predictions); result=va[['StudyInstanceUID','fold']].copy()
        for i,target in enumerate(TARGETS): result[target]=pred[:,i]
        result.to_csv(OUT/'validation_predictions.csv',index=False)
pd.DataFrame(history).to_csv(OUT/'history.csv',index=False)
metrics=[]; saved=pd.read_csv(OUT/'validation_predictions.csv')
for i,target in enumerate(TARGETS):
    mask=(va[wc[i]].values>0)&bool(active[i].item()); y=va[yc[i]].values[mask]; p=saved[target].values[mask]
    metrics.append({'target':target,'n_supervised':len(y),'auc':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None})
pd.DataFrame(metrics).to_csv(OUT/'target_metrics.csv',index=False); display(pd.DataFrame(metrics))
print('Saved',OUT,'Best masked BCE',best,'Missing AUC means no supervised positive/negative pair, or no training supervision.')